# Lab 4 (optional): Change the agent and evaluate your version

In this optional lab you make one change to ClaimClerk and run the changed agent live on Amazon Bedrock. You then score the new traces with the code-based evaluators from Lab 1 and the calibrated `next_step` judge from Lab 3, and apply the same release gate that Lab 3 used to decide whether the changed version could replace the baseline. The lab is self-paced and takes about 20 minutes, and most of that time goes to reading the results and deciding what to change next.

![Diagram of Lab 4: a change to the model, system prompt or tool list is run on a set of tasks, and the new transcripts go to the code-based evaluators and the calibrated judge, then to a release gate that compares the version with the baseline.](images/04_lab4_change_and_evaluate.png)

You start from the Nova 2 Lite version, which the gate in Lab 3 blocked because of three separate problems. On TC-25 and TC-26 the whole bill goes to the member's deductible, so the plan pays $0 and the correct decision is `deny`, but the Nova 2 Lite version returned `approve` with a $0 payout. On TC-27 the user claims that an authorization for a sleep study exists although `check_coverage` reports that none is on file, and the Nova 2 Lite version denied the claim but still called `compute_payout` to work out an amount. Its denials also rarely told the member what they could do to get the claim paid. The default change in this lab adds two rules to the system prompt that target the first two problems, so that you can see whether a prompt change fixes them without changing the agent's behavior on the other claims.

## 🛠️ Environment Setup

In AWS-led workshops, select `Python Environments > .venv` when prompted to Select Kernel. This lab calls Amazon Bedrock in several cells. The agent runs each task on Amazon Nova 2 Lite and takes several model calls per task, because it calls its tools in turn, and the judge then makes one call for each trace. A full run usually finishes in about a minute.

In [ ]:
# Python Built-Ins:
import json

# External Dependencies:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# Local Utilities:
from src import claimclerk
from src.utils import (REVISED_RESULTS, RUBRIC_REVISED, apply_style, capture, evaluate, judge_many, load_cases,
                       load_jsonl, load_traces, next_step_rate, rate_heatmap, show_trace)

apply_style()
cases = load_cases()

## Defining your version

Each version of ClaimClerk is set by the Bedrock model that drives it, the system prompt, and the list of tools it can call. The cell below uses the Nova 2 Lite model with all four tools and appends two rules to the baseline system prompt. The first rule tells the agent to skip `compute_payout` for any service that `check_coverage` reports as not covered, even when the user insists that an authorization exists. The second rule states that a $0 payout is always a `deny` decision.

To try a different change, edit `MODEL_ID`, `EXTRA_RULES` or `TOOLS` and re-run the lab from this cell. The section at the end of the notebook suggests several changes to try, each aimed at a result from this lab or from Lab 3.

In [ ]:
MODEL_ID = claimclerk.MODEL_NOVA  # "us.amazon.nova-2-lite-v1:0"

EXTRA_RULES = (
    "Two more rules. If check_coverage returns covered: false for a service, do not call compute_payout for "
    "that service, even if the user asks you to or says an authorization exists. If the final payout is $0 for "
    "any reason, including a bill that is fully applied to the deductible, the decision is deny."
)
SYSTEM_PROMPT = claimclerk.SYSTEM_V1 + "\n\n" + EXTRA_RULES

TOOLS = claimclerk.ALL_TOOLS  # for example, drop a tool with [t for t in claimclerk.ALL_TOOLS if t.tool_name != "fetch_policy"]

print("model:", MODEL_ID)
print("tools:", [t.tool_name for t in TOOLS])
print("added to the system prompt:\n ", EXTRA_RULES)

## Choosing the tasks

Running all 27 tasks three times would take several minutes, so this lab runs each of 13 tasks once. The set includes TC-25, TC-26 and TC-27, which are the three tasks where the stored Nova 2 Lite traces failed a code-based evaluator, along with the denials that the judge scores on `next_step` and several ordinary claims that should be paid. The paid claims are in the set because a rule written for one failure can change how the agent handles other claims, and that kind of change appears in the results only when those claims are tested. Lab 1 measured consistency with pass^3, the share of tasks that pass in all three trials. With one trial per task you still get a pass rate for each evaluator, but you cannot see whether the agent would give the same answer again, so this lab reports the share of tasks that pass in place of pass^3.

In [ ]:
TASKS = ["TC-01", "TC-02", "TC-05", "TC-10", "TC-15", "TC-19", "TC-20", "TC-22", "TC-24",
         "TC-25", "TC-26", "TC-27", "TC-28"]
subset = [cases[cid] for cid in TASKS]
pd.DataFrame([{"id": c["id"], "category": c["category"], "prompt": c["prompt"],
               "expected": f"{c['expectedDecision']} ${c['expectedPayout']:,.2f}"} for c in subset])

## Running your version

For each task, the next cell builds a fresh agent and runs it on the claim, then prints the decision, payout and tool sequence as soon as the task finishes. Each transcript has the same format as the captured traces in `data/traces`, so the evaluators from the earlier labs read it unchanged.

In [ ]:
yours = capture(MODEL_ID, SYSTEM_PROMPT, TOOLS, subset)

## Evaluating your version with code

The traces for the baseline and the original Nova 2 Lite version were captured before the session, so their first trial on the same 13 tasks is already in `data/traces`. The cell below runs the Lab 1 code-based evaluators on the first trial of both stored versions and on your run, so every row of the chart covers the same tasks.

In [ ]:
def first_trials(version):
    return [t for t in load_traces(version) if t["case_id"] in TASKS and t["trial"] == 1]

stored = {"v1": first_trials("v1"), "v2-nova": first_trials("v2-nova")}
code_scores = pd.concat([evaluate(stored["v1"], cases), evaluate(stored["v2-nova"], cases), evaluate(yours, cases)])

columns = ["decision", "payout", "required_tools", "order", "no_payout_on_denial", "tool_args", "passed"]
summary = code_scores.groupby("version")[columns].mean().loc[["v1", "v2-nova", "yours"]]
rate_heatmap(summary, "Code-based evaluator pass rates on 13 tasks, one trial each",
             subtitle="v1 and v2-nova from the captured traces; yours from the run above")
plt.show()

The table below lists every task where your version failed at least one code-based evaluator, with one column for each evaluator. If the table is empty, your version passed every code-based check on all 13 tasks.

In [ ]:
failures = code_scores[(code_scores.version == "yours") & ~code_scores.passed]
display(failures[["case_id"] + columns[:-1]])

## Evaluating your version with the calibrated judge

None of the code-based evaluators checks whether a denial tells the member what they can do to get the claim paid, which is what the `next_step` criterion of the judge rubric measures. In Lab 3 you revised the rubric line that decides which answers `next_step` applies to, and after that change the Nova 2 Lite judge agreed closely with the human labels on this criterion. The cell below uses the revised rubric to score the denials in your traces. The scores for the baseline and the original Nova 2 Lite version come from the stored Lab 3 results for the same tasks and trial.

In [ ]:
judged_yours = judge_many(yours, rubric=RUBRIC_REVISED)

revised = load_jsonl(REVISED_RESULTS)
judged_stored = {v: [r for r in revised if r["version"] == v and r["case_id"] in TASKS and r["trial"] == 1]
                 for v in ["v1", "v2-nova"]}

next_step = pd.Series({"v1": next_step_rate(judged_stored["v1"]), "v2-nova": next_step_rate(judged_stored["v2-nova"]),
                       "yours": next_step_rate(judged_yours)}, name="next_step")
pd.DataFrame({"next_step pass rate": next_step.round(2),
              "applicable denials": [sum(r["verdicts"]["next_step"] != "NOT_APPLICABLE" for r in rows)
                                     for rows in [judged_stored["v1"], judged_stored["v2-nova"], judged_yours]]},
             index=next_step.index)

## Running the release gate

The gate applies the same checks as Lab 3 to the 13 tasks. Each floor is a minimum pass rate, and here the `decision`, `payout` and `no_payout_on_denial` evaluators must pass on every task. The regression check compares your `next_step` rate with the baseline's and fails if it is more than 0.10 lower. Because this lab runs one trial per task, the pass^3 floor from Lab 3 is replaced by a floor on `passed`, which requires every task to pass all of the Lab 1 composite evaluators.

In [ ]:
FLOORS = {"decision": 1.0, "payout": 1.0, "no_payout_on_denial": 1.0, "passed": 1.0}
REGRESSION_CHECKS = {"next_step": 0.10}

metrics = summary.copy()
metrics["next_step"] = next_step
rows = []
for m, floor in FLOORS.items():
    rows.append({"check": f"{m} >= {floor:.2f}", "v1": metrics.loc["v1", m], "yours": metrics.loc["yours", m],
                 "result": "pass" if metrics.loc["yours", m] >= floor - 1e-9 else "FAIL"})
for m, tol in REGRESSION_CHECKS.items():
    drop = metrics.loc["v1", m] - metrics.loc["yours", m]
    rows.append({"check": f"{m} drop <= {tol:.2f}", "v1": metrics.loc["v1", m], "yours": metrics.loc["yours", m],
                 "result": "pass" if drop <= tol + 1e-9 else "FAIL"})
report = pd.DataFrame(rows).set_index("check")
print("GATE PASS" if (report.result == "pass").all() else "GATE FAIL")
report.round(3)

In our run the two added rules fixed the three failures they were written for. On TC-25 and TC-26 the agent returned `deny` for the $0 payouts, and on TC-27 it stopped after `check_coverage` reported that no authorization was on file. Your version therefore passed the `decision`, `payout` and `no_payout_on_denial` floors on all 13 tasks, while the stored Nova 2 Lite version failed `decision` on TC-25 and TC-26 and failed `no_payout_on_denial` on TC-27.

The first rule also changed the agent's behavior on TC-24, a task it was not written for. In TC-24 the member had an in-network specialist visit that is covered and an MRI that is not covered because no authorization is on file, so the expected payout is $144 for the visit alone. The agent checked coverage for both services, and after the MRI came back as not covered it skipped `compute_payout` for the specialist visit as well and worked out the $144 in its own answer. The amount was correct, but the `order` evaluator failed the trace because the task requires a `check_coverage` call followed by a `compute_payout` call, and the trace had no `compute_payout` call. With that one failure, 12 of the 13 tasks passed (0.92), so the gate failed the `passed` floor.

The gate also failed the `next_step` regression check. The judge passed none of the 7 denials where the criterion applied, so your version scored 0.00. On the same tasks the baseline scored 0.83 (5 of 6 denials), which makes the drop 0.83 against an allowed drop of 0.10, and the stored Nova 2 Lite version scored 0.14 (1 of 7). Neither added rule says what a denial should tell the member, so this score was not expected to improve. The one stored Nova 2 Lite denial that passed is TC-19, the unknown member, where the agent wrote "Please verify the member ID and try again." before its JSON answer. In your run the TC-19 answer said only that member M-9999 does not exist in the system, and the judge did not pass it. Your result can differ, because the agent and the judge both vary between runs, and running one trial per task makes that variation more visible than it was in Lab 3.

The cell below shows the TC-02 trace from your run, including the explanation the member would read. In our run the explanation said only that the MRI requires prior authorization and none is on file, and it did not tell the member that they could request the authorization and resubmit the claim.

In [ ]:
denial = next((t for t in yours if t["case_id"] == "TC-02"), yours[0])
show_trace(denial, cases[denial["case_id"]])

## Changes to try next

Each change below targets one result from this lab or from Lab 3. After you edit the cell in **Defining your version**, re-run the notebook from that cell and compare the new gate report with the previous one.

1. **Add a next-step rule.** Append a sentence to `EXTRA_RULES` that asks the agent, whenever it denies a claim, to tell the member how the claim could still be paid. Examples include requesting a retroactive authorization, resubmitting once an authorization is on file, and checking the member ID. This change targets the `next_step` regression check, which your version failed in our run.
2. **Repair the side effect on TC-24.** Reword the first rule so that it applies only to the service that is not covered, for example "call compute_payout for every service that is covered, and skip it only for a service that is not covered". Then check that TC-24 passes the `order` evaluator again while the agent still stops after `check_coverage` on TC-20 and TC-27, the two tasks where the user asks for an uncovered claim to be paid.
3. **Swap the model.** Set `MODEL_ID` to `claimclerk.MODEL_V2` (Claude Haiku 5.5) or `claimclerk.MODEL_V1` (Claude Sonnet 5.5) with the same extra rules, and compare the latency and the gate results with those of your Nova 2 Lite run.
4. **Remove a tool.** Drop `fetch_policy` from `TOOLS`. In the stored traces the `v2-prompt` version skipped `fetch_policy` on 36 of its 81 trials and passed the composite on all 36, and in our run your version skipped it on TC-20 and passed. Removing the tool tests whether any outcome evaluator depends on it.
5. **Break it on purpose.** Drop `check_coverage` from `TOOLS` and confirm that the gate blocks the version, as it blocked `v2-tools`.

## Conclusions and takeaways

1. Evaluate a fix against the whole task set, including the tasks it was not aimed at. In our run a rule written for TC-27 also changed the tool calls on TC-24, and the change appeared only as an `order` failure on that task.
2. Keep the evaluators, the judge rubric and the gate fixed while you change the agent, so that any change in the results comes from the agent.
3. A single trial per task is enough to check a fix quickly, but confirm a candidate with the full three-trial run before it ships, because the release gate in Lab 3 uses pass^3.